# 🔍 Agent Reliability Lab
### Stop confusing 'conversation completion' with actual task success

> **The problem:** Your agent reports 'Updated 47 records ✓' - the database shows 0 changes.
> LLMs complete conversations helpfully. When tool calls fail silently, you find out weeks later.

| Pattern | What it catches |
|---------|----------------|
| 1. Verification Tools | Silent failures, async jobs reported as done |
| 2. Explicit Success Criteria | Vague success from ambiguous API responses |
| 3. Action Event Logging | Gaps between attempted -> completed -> verified |
| 4. Idempotency Tokens | Duplicate writes from agent retries |

> Make sure **Ollama is running** (`ollama serve`) and you have pulled your chosen model
> before running. Default model: `llama3.2`. Change `MODEL` in Section 0 to any model you have.


## Section 0 - Setup & Simulated Infrastructure

In [3]:

!pip install ollama pydantic -q

import os, time, json, uuid, hashlib, random
from datetime import datetime, timezone
from dataclasses import dataclass, field
from typing import Optional
from enum import Enum
from pydantic import BaseModel

import ollama

MODEL = "llama3.2"   # <- change to any model you have pulled, e.g. mistral, phi3, gemma3

def call_ollama(prompt: str, system: str = "", max_tokens: int = 512) -> str:
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    else:
        messages.append({"role": "system", "content": "You are a helpful assistant."})
    messages.append({"role": "user", "content": prompt})
    response = ollama.chat(
        model=MODEL,
        messages=messages,
        options={"num_predict": max_tokens}
    )
    return response["message"]["content"]

# ── In-memory customer DB (ground truth) ──────────────────────────────────────
CUSTOMER_DB: dict[str, dict] = {
    "C001": {"id": "C001", "name": "Alice Chen",  "email": "alice@example.com",  "plan": "starter"},
    "C002": {"id": "C002", "name": "Bob Smith",   "email": "bob@example.com",    "plan": "pro"},
    "C003": {"id": "C003", "name": "Carol White", "email": "carol@example.com",  "plan": "starter"},
}
IDEMPOTENCY_STORE: dict[str, dict] = {}
ORDERS_DB: list[dict] = []

def reset_db():
    CUSTOMER_DB["C001"]["email"] = "alice@example.com"
    CUSTOMER_DB["C002"]["email"] = "bob@example.com"
    CUSTOMER_DB["C003"]["email"] = "carol@example.com"
    for d in [CUSTOMER_DB["C001"], CUSTOMER_DB["C002"], CUSTOMER_DB["C003"]]:
        d["plan"] = "starter" if d["id"] != "C002" else "pro"
    ORDERS_DB.clear()
    IDEMPOTENCY_STORE.clear()

def db_get(cid: str) -> Optional[dict]:
    return CUSTOMER_DB.get(cid)

def db_write(cid: str, updates: dict) -> bool:
    if cid not in CUSTOMER_DB:
        return False
    CUSTOMER_DB[cid].update(updates)
    return True

reset_db()
print("Infrastructure ready. Customers:", list(CUSTOMER_DB.keys()))
print(f"Using model: {MODEL}  (run 'ollama pull {MODEL}' if not already downloaded)")


zsh:1: command not found: pip
Infrastructure ready. Customers: ['C001', 'C002', 'C003']
Using model: llama3.2  (run 'ollama pull llama3.2' if not already downloaded)


---
## Section 1 - Verification Tools

**The pattern:** After every state-changing call, the agent MUST call a `verify_*` tool that reads ground truth. No success report until verification passes.

**Three failure modes we simulate:**
- `ASYNC_QUEUED` - API returns 200 with `{status: queued}`, background job never runs
- `SILENT_CORRUPT` - API returns 200 OK, but writes garbage to DB
- `SUCCESS` - Normal, verify confirms it

Watch: unverified agent reports success in ALL three. Verified agent catches the failures.

In [ ]:

class FailureMode(Enum):
    SUCCESS        = "success"
    ASYNC_QUEUED   = "async_queued"
    SILENT_CORRUPT = "silent_corrupt"

def api_update_email(cid: str, new_email: str, mode: FailureMode) -> dict:
    '''Simulates external API. Returns what it *says* happened.'''
    if mode == FailureMode.SUCCESS:
        db_write(cid, {"email": new_email})
        return {"status": "ok", "message": "Email updated successfully"}
    elif mode == FailureMode.ASYNC_QUEUED:
        return {"status": "queued", "message": "Update queued for processing"}
    elif mode == FailureMode.SILENT_CORRUPT:
        db_write(cid, {"email": "corrupted@error.internal"})
        return {"status": "ok", "message": "Email updated successfully"}

def tool_verify_email(cid: str, expected: str) -> dict:
    '''Reads ground truth. Returns unambiguous verified=True/False.'''
    rec = db_get(cid)
    if not rec:
        return {"verified": False, "reason": f"{cid} not found"}
    actual = rec["email"]
    return {"verified": actual == expected, "actual": actual, "expected": expected}

def unverified_agent(cid: str, email: str, mode: FailureMode) -> str:
    '''Trusts API response, never checks DB.'''
    api = api_update_email(cid, email, mode)
    prompt = (f"You called update_email for {cid} -> '{email}'. "
              f"API returned: {api}. Report the outcome in 1-2 sentences.")
    return call_ollama(prompt)

def verified_agent(cid: str, email: str, mode: FailureMode) -> str:
    '''Calls verify tool before reporting. Explicit success criteria.'''
    api    = api_update_email(cid, email, mode)
    verify = tool_verify_email(cid, email)
    prompt = (
        "Task: update email for {cid} to '{email}'."
        "API response:    {api}"
        "Verification:    {verify}"
        "RULE: You have NOT completed this task until verified=True. "
        "If verified=False, report FAILURE and state what the DB actually contains."
    )
    return call_ollama(prompt)

# ── Run all three modes ────────────────────────────────────────────────────────
tests = [
    (FailureMode.SUCCESS,        "C001", "alice_new@example.com"),
    (FailureMode.ASYNC_QUEUED,   "C002", "bob_new@example.com"),
    (FailureMode.SILENT_CORRUPT, "C003", "carol_new@example.com"),
]

print("=" * 62)
print("VERIFICATION TOOLS DEMO")
print("=" * 62)

for mode, cid, email in tests:
    print(f"
--- Mode: {mode.value.upper()} | {cid} -> {email} ---")
    reset_db()

    # Without verification
    result  = unverified_agent(cid, email, mode)
    actual  = db_get(cid)["email"]
    match   = "✅" if actual == email else "❌ MISMATCH"
    print(f"[NO VERIFY] Agent: {result.strip()[:120]}")
    print(f"            DB: '{actual}'  {match}")

    reset_db()

    # With verification
    result  = verified_agent(cid, email, mode)
    actual  = db_get(cid)["email"]
    honest  = actual == email or any(w in result.lower() for w in ["fail","incomplete","not","error"])
    print(f"[VERIFIED]  Agent: {result.strip()[:120]}")
    print(f"            DB: '{actual}'  {'✅ caught' if not (actual==email) and honest else '✅'}")


SyntaxError: unterminated f-string literal (detected at line 37) (2494689673.py, line 37)


def api_upgrade_plan(cid: str, new_plan: str) -> dict:
    '''Returns 200 but intentionally does NOT write to DB (simulates async queue).'''
    return {"status": 200, "message": "Plan upgrade request received",
            "request_id": uuid.uuid4().hex[:8]}

def tool_verify_plan(cid: str, expected: str) -> dict:
    rec = db_get(cid)
    actual = rec["plan"] if rec else None
    return {"verified": actual == expected, "actual_plan": actual, "expected_plan": expected}

VAGUE_SYSTEM = "You are a customer success agent. Update the customer plan when requested."

EXPLICIT_SYSTEM = (
    "You are a customer success agent with strict verification requirements.\n\n"
    "SUCCESS CRITERIA - a plan upgrade is complete ONLY when:\n\n"
    "  1. The API returned a non-error response\n\n"
    "  2. verify_plan() returns verified=True with the correct plan name\n\n"
    "A 200 status or 'request received' is NOT proof of completion.\n\n"
    "If verify_plan() returns verified=False, you MUST report the task as INCOMPLETE "
    "and include what the DB actually contains."
)

def run_prompt_style(system: str, label: str, cid: str, new_plan: str):
    reset_db()
    api    = api_upgrade_plan(cid, new_plan)
    verify = tool_verify_plan(cid, new_plan)
    prompt = (f"Task: upgrade {cid} to '{new_plan}' plan.\n"
              f"API response: {json.dumps(api)}\n"
              f"verify_plan() result: {json.dumps(verify)}\n\n"
              "Report the outcome.")
    resp   = call_ollama(prompt, system=system)
    actual = db_get(cid)["plan"]
    honest = actual == new_plan or any(w in resp.lower()
             for w in ["incomplete","fail","not complete","not updated","did not","unsuccessful"])
    print(f"\n[{label}]")
    print(f"  Agent says : {resp.strip()[:160]}")
    print(f"  DB reality : plan='{actual}' (expected '{new_plan}')")
    print(f"  Honest?    : {'✅ yes' if honest else '⚠️  agent may be over-optimistic'}")

print("=" * 62)
print("EXPLICIT SUCCESS CRITERIA DEMO")
print("Scenario: plan upgrade API queues change but never applies it")
print("=" * 62)

run_prompt_style(VAGUE_SYSTEM,    "VAGUE PROMPT",    "C001", "pro")
run_prompt_style(EXPLICIT_SYSTEM, "EXPLICIT PROMPT", "C001", "pro")


In [ ]:

def api_upgrade_plan(cid: str, new_plan: str) -> dict:
    '''Returns 200 but intentionally does NOT write to DB (simulates async queue).'''
    return {"status": 200, "message": "Plan upgrade request received",
            "request_id": uuid.uuid4().hex[:8]}

def tool_verify_plan(cid: str, expected: str) -> dict:
    rec = db_get(cid)
    actual = rec["plan"] if rec else None
    return {"verified": actual == expected, "actual_plan": actual, "expected_plan": expected}

VAGUE_SYSTEM = "You are a customer success agent. Update the customer plan when requested."

EXPLICIT_SYSTEM = (
    "You are a customer success agent with strict verification requirements.

"
    "SUCCESS CRITERIA - a plan upgrade is complete ONLY when:
"
    "  1. The API returned a non-error response
"
    "  2. verify_plan() returns verified=True with the correct plan name

"
    "A 200 status or 'request received' is NOT proof of completion.
"
    "If verify_plan() returns verified=False, you MUST report the task as INCOMPLETE "
    "and include what the DB actually contains."
)

def run_prompt_style(system: str, label: str, cid: str, new_plan: str):
    reset_db()
    api    = api_upgrade_plan(cid, new_plan)
    verify = tool_verify_plan(cid, new_plan)
    prompt = (f"Task: upgrade {cid} to '{new_plan}' plan.
"
              f"API response: {json.dumps(api)}
"
              f"verify_plan() result: {json.dumps(verify)}

"
              "Report the outcome.")
    resp   = call_ollama(prompt, system=system)
    actual = db_get(cid)["plan"]
    honest = actual == new_plan or any(w in resp.lower()
             for w in ["incomplete","fail","not complete","not updated","did not","unsuccessful"])
    print(f"
[{label}]")
    print(f"  Agent says : {resp.strip()[:160]}")
    print(f"  DB reality : plan='{actual}' (expected '{new_plan}')")
    print(f"  Honest?    : {'✅ yes' if honest else '⚠️  agent may be over-optimistic'}")

print("=" * 62)
print("EXPLICIT SUCCESS CRITERIA DEMO")
print("Scenario: plan upgrade API queues change but never applies it")
print("=" * 62)

run_prompt_style(VAGUE_SYSTEM,    "VAGUE PROMPT",    "C001", "pro")
run_prompt_style(EXPLICIT_SYSTEM, "EXPLICIT PROMPT", "C001", "pro")


---
## Section 3 - Separate Action Event Logging

**The pattern:** Every action emits three distinct events:

```
action_attempted  ->  action_completed  ->  action_verified
```

**Alert on gaps:**
- `completed` without `verified` → agent never checked ground truth
- `attempted` without `completed` → tool call crashed
- `verified=False` after `completed` → silent failure (the dangerous one)

We run a batch of 6 updates with mixed failure modes and show the gap report.

In [ ]:
---
## Section 5 - Full Reliable Agent

Combines all four patterns into a `ReliableAgent` class:

```
For every action:
  1. Generate idempotency key
  2. Log action_attempted
  3. Execute (with idempotency check)
  4. Log action_completed
  5. Call verify_* tool
  6. Log action_verified / action_failed
  7. Ask the LLM to report - with EXPLICIT success criteria
```

The final report shows **true success rate** vs conversation completion rate.



RELIABLE_SYSTEM = (
    "You are a reliable customer management agent.\n\n"
    "STRICT RULES:\n"
    "1. You have NOT completed a task until verification shows verified=True.\n"
    "2. If verified=False, report INCOMPLETE and state what the DB actually contains.\n"
    "3. Never treat 'queued', 'pending', or 'received' as success.\n"
    "4. Be concise - 2 sentences max."
)

class ReliableAgent:
    def __init__(self):
        self.logger   = ActionLogger()
        self.session  = f"sess_{uuid.uuid4().hex[:8]}"
        self.outcomes: list[dict] = []

    def update_email(self, cid: str, new_email: str,
                     mode: FailureMode = FailureMode.SUCCESS) -> dict:
        aid      = f"act_{uuid.uuid4().hex[:6]}"
        idem_key = make_key("update_email", cid, new_email, self.session)

        # 1. Log attempt
        self.logger.log(aid, "update_email", ActionStatus.ATTEMPTED,
                        {"cid": cid, "new_email": new_email, "idem_key": idem_key})

        # 2. Idempotency check
        if idem_key in IDEMPOTENCY_STORE:
            self.logger.log(aid, "update_email", ActionStatus.COMPLETED,
                            {"status": "replayed"})
            return {"action_id": aid, "status": "replayed"}

        # 3. Execute
        try:
            api = api_update_email(cid, new_email, mode)
            IDEMPOTENCY_STORE[idem_key] = api
            self.logger.log(aid, "update_email", ActionStatus.COMPLETED,
                            {"api_status": api.get("status")})
        except Exception as ex:
            self.logger.log(aid, "update_email", ActionStatus.FAILED, {"error": str(ex)})
            return {"action_id": aid, "status": "error"}

        # 4. Verify
        verify  = tool_verify_email(cid, new_email)
        vstatus = ActionStatus.VERIFIED if verify["verified"] else ActionStatus.FAILED
        self.logger.log(aid, "update_email", vstatus, verify)

        # 5. LLM report with explicit criteria
        prompt  = (f"Task: update email for {cid} to '{new_email}'.\n"
                   f"API response: {json.dumps(api)}\n"
                   f"Verification: {json.dumps(verify)}\n\nReport the outcome.")
        report  = call_ollama(prompt, system=RELIABLE_SYSTEM)

        outcome = {"action_id": aid, "cid": cid, "new_email": new_email,
                   "api_status": api.get("status"), "verified": verify["verified"],
                   "actual_db": verify.get("actual"), "report": report.strip()}
        self.outcomes.append(outcome)
        return outcome

    def final_report(self):
        total  = len(self.outcomes)
        ok     = sum(1 for o in self.outcomes if o.get("verified"))
        gaps   = self.logger.gap_report()
        print(f"\n{'='*60}")
        print("RELIABLE AGENT - FINAL REPORT")
        print(f"{'='*60}")
        print(f"Actions attempted  : {total}")
        print(f"Truly verified  ✅ : {ok}")
        print(f"Silent failures ❌ : {total - ok}")
        print(f"True success rate  : {ok/total*100:.1f}%  <- real metric")
        print(f"(Conversation completion rate would show ~100%)")
        if gaps:
            print(f"\n⚠️  {len(gaps)} gaps logged for investigation:")
            for g in gaps: print(f"  {g['id']} -- {g['gap']}")
        print("\nPer-action breakdown:")
        for o in self.outcomes:
            icon = "✅" if o.get("verified") else "❌"
            print(f"  {icon} {o['action_id']} | {o['cid']} | api={o['api_status']} | "
                  f"db='{o.get('actual_db')}' | {o['report'][:80]}...")

# ── Run across all failure modes ──────────────────────────────────────────────
reset_db()

agent = ReliableAgent()

tasks = [
    ("C001", "alice_final@example.com", FailureMode.SUCCESS),
    ("C002", "bob_final@example.com",   FailureMode.ASYNC_QUEUED),
    ("C003", "carol_final@example.com", FailureMode.SILENT_CORRUPT),
    ("C001", "alice_final@example.com", FailureMode.SUCCESS),   # retry -> idempotent
    ("C002", "bob_ok@example.com",      FailureMode.SUCCESS),   # new email -> new action
]

print("=" * 60)
print("RELIABLE AGENT - 5 actions (mixed failure modes)")
print("=" * 60)

for cid, email, mode in tasks:
    print(f"\n>>> {cid} -> {email} [{mode.value}]")
    agent.update_email(cid, email, mode)

agent.final_report()


In [ ]:

def make_key(*args) -> str:
    raw = ":".join(str(a) for a in args)
    return hashlib.sha256(raw.encode()).hexdigest()[:16]

def create_order_raw(cid: str, amount: float, desc: str) -> dict:
    '''No idempotency - every call creates a new order.'''
    order = {"order_id": f"ORD-{uuid.uuid4().hex[:6].upper()}",
             "customer_id": cid, "amount": amount, "description": desc,
             "created_at": datetime.now(timezone.utc).isoformat()}
    ORDERS_DB.append(order)
    return {"status": "created", "order": order}

def create_order_safe(cid: str, amount: float, desc: str, session: str) -> dict:
    '''Idempotent: same inputs in same session always return same result.'''
    key = make_key("create_order", cid, amount, desc, session)
    if key in IDEMPOTENCY_STORE:
        return {"status": "replayed", "order": IDEMPOTENCY_STORE[key]["order"],
                "key": key, "note": "duplicate - returning cached result"}
    result = create_order_raw(cid, amount, desc)
    IDEMPOTENCY_STORE[key] = result
    result["key"] = key
    return result

print("=" * 60)
print("IDEMPOTENCY TOKEN DEMO")
print("=" * 60)

# Without idempotency
ORDERS_DB.clear(); IDEMPOTENCY_STORE.clear()
print("
--- WITHOUT idempotency (agent retries 3x on timeout) ---")
for i in range(1, 4):
    r = create_order_raw("C001", 299.99, "Pro plan upgrade")
    print(f"  Attempt {i}: order_id={r['order']['order_id']}  status={r['status']}")
print(f"  Orders in DB: {len(ORDERS_DB)}  <- 3 DUPLICATE ORDERS 🚨")

# With idempotency
ORDERS_DB.clear(); IDEMPOTENCY_STORE.clear()
SESSION = "sess_abc123"
print("
--- WITH idempotency (same agent, same session, 3 retries) ---")
for i in range(1, 4):
    r = create_order_safe("C001", 299.99, "Pro plan upgrade", SESSION)
    print(f"  Attempt {i}: order_id={r['order']['order_id']}  status={r['status']}  key={r.get('key','')}")
print(f"  Orders in DB: {len(ORDERS_DB)}  <- exactly 1 ✅")

# Key stability
print("
--- Key stability check ---")
keys = [make_key("create_order", "C001", 299.99, "Pro plan upgrade", SESSION) for _ in range(5)]
print(f"  Same inputs -> same key every time : {len(set(keys)) == 1} ✅  key={keys[0]}")
other = make_key("create_order", "C001", 299.99, "Pro plan upgrade", "sess_OTHER")
print(f"  Different session -> different key : {other != keys[0]} ✅")


---
## Section 5 - Full Reliable Agent

Combines all four patterns into a `ReliableAgent` class:

```
For every action:
  1. Generate idempotency key
  2. Log action_attempted
  3. Execute (with idempotency check)
  4. Log action_completed
  5. Call verify_* tool
  6. Log action_verified / action_failed
  7. Ask the LLM to report - with EXPLICIT success criteria
```

The final report shows **true success rate** vs conversation completion rate.

In [ ]:

RELIABLE_SYSTEM = (
    "You are a reliable customer management agent.

"
    "STRICT RULES:
"
    "1. You have NOT completed a task until verification shows verified=True.
"
    "2. If verified=False, report INCOMPLETE and state what the DB actually contains.
"
    "3. Never treat 'queued', 'pending', or 'received' as success.
"
    "4. Be concise - 2 sentences max."
)

class ReliableAgent:
    def __init__(self):
        self.logger   = ActionLogger()
        self.session  = f"sess_{uuid.uuid4().hex[:8]}"
        self.outcomes: list[dict] = []

    def update_email(self, cid: str, new_email: str,
                     mode: FailureMode = FailureMode.SUCCESS) -> dict:
        aid      = f"act_{uuid.uuid4().hex[:6]}"
        idem_key = make_key("update_email", cid, new_email, self.session)

        # 1. Log attempt
        self.logger.log(aid, "update_email", ActionStatus.ATTEMPTED,
                        {"cid": cid, "new_email": new_email, "idem_key": idem_key})

        # 2. Idempotency check
        if idem_key in IDEMPOTENCY_STORE:
            self.logger.log(aid, "update_email", ActionStatus.COMPLETED,
                            {"status": "replayed"})
            return {"action_id": aid, "status": "replayed"}

        # 3. Execute
        try:
            api = api_update_email(cid, new_email, mode)
            IDEMPOTENCY_STORE[idem_key] = api
            self.logger.log(aid, "update_email", ActionStatus.COMPLETED,
                            {"api_status": api.get("status")})
        except Exception as ex:
            self.logger.log(aid, "update_email", ActionStatus.FAILED, {"error": str(ex)})
            return {"action_id": aid, "status": "error"}

        # 4. Verify
        verify  = tool_verify_email(cid, new_email)
        vstatus = ActionStatus.VERIFIED if verify["verified"] else ActionStatus.FAILED
        self.logger.log(aid, "update_email", vstatus, verify)

        # 5. LLM report with explicit criteria
        prompt  = (f"Task: update email for {cid} to '{new_email}'.
"
                   f"API response: {json.dumps(api)}
"
                   f"Verification: {json.dumps(verify)}

Report the outcome.")
        report  = call_ollama(prompt, system=RELIABLE_SYSTEM)

        outcome = {"action_id": aid, "cid": cid, "new_email": new_email,
                   "api_status": api.get("status"), "verified": verify["verified"],
                   "actual_db": verify.get("actual"), "report": report.strip()}
        self.outcomes.append(outcome)
        return outcome

    def final_report(self):
        total  = len(self.outcomes)
        ok     = sum(1 for o in self.outcomes if o.get("verified"))
        gaps   = self.logger.gap_report()
        print(f"
{'='*60}")
        print("RELIABLE AGENT - FINAL REPORT")
        print(f"{'='*60}")
        print(f"Actions attempted  : {total}")
        print(f"Truly verified  ✅ : {ok}")
        print(f"Silent failures ❌ : {total - ok}")
        print(f"True success rate  : {ok/total*100:.1f}%  <- real metric")
        print(f"(Conversation completion rate would show ~100%)")
        if gaps:
            print(f"
⚠️  {len(gaps)} gaps logged for investigation:")
            for g in gaps: print(f"  {g['id']} -- {g['gap']}")
        print("
Per-action breakdown:")
        for o in self.outcomes:
            icon = "✅" if o.get("verified") else "❌"
            print(f"  {icon} {o['action_id']} | {o['cid']} | api={o['api_status']} | "
                  f"db='{o.get('actual_db')}' | {o['report'][:80]}...")

# ── Run across all failure modes ──────────────────────────────────────────────
reset_db()

agent = ReliableAgent()

tasks = [
    ("C001", "alice_final@example.com", FailureMode.SUCCESS),
    ("C002", "bob_final@example.com",   FailureMode.ASYNC_QUEUED),
    ("C003", "carol_final@example.com", FailureMode.SILENT_CORRUPT),
    ("C001", "alice_final@example.com", FailureMode.SUCCESS),   # retry -> idempotent
    ("C002", "bob_ok@example.com",      FailureMode.SUCCESS),   # new email -> new action
]

print("=" * 60)
print("RELIABLE AGENT - 5 actions (mixed failure modes)")
print("=" * 60)

for cid, email, mode in tasks:
    print(f"
>>> {cid} -> {email} [{mode.value}]")
    agent.update_email(cid, email, mode)

agent.final_report()
